# Trim half of train and validation dataset.

In [ ]:
import os
import json
import numpy as np

# Some constants:
ROOT_DIR = os.getcwd() # /content folder in gdrive.
GDRIVE_FOLDER_PATH = os.path.join(ROOT_DIR, "drive/MyDrive/Colab Notebooks/COMP592_Transaprent_Obj_Detection")
DATASET_DIR = os.path.join(GDRIVE_FOLDER_PATH, "data")



In [19]:
def getAnnotations(mode):
  # assert mode in ["train", "val"]
  dataset_dir = os.path.join(DATASET_DIR, mode)
  annotation_file_path = os.path.join(dataset_dir, "region_data_"+mode+".json")
  # annotation_file_path = os.path.join(dataset_dir, "region_data_"+mode+"_single_imgs.json")
  annotations = json.load(open(annotation_file_path))
  if type(annotations) == dict:
    annotations = list(annotations.values())
  annotations = [a for a in annotations if a['regions']]
  print(len(annotations))
  return annotations

annotations_train = getAnnotations("train")
annotations_val = getAnnotations("val")
# annotations = annotations_train + annotations_val
# np.random.shuffle(annotations)

# print(len(annotations_train), len(annotations_val), len(annotations))

337
140
137


## Find number of each object type in train, val

In [20]:
def getDictionaryWithObjectTypes():
  return {"data": None, "Glass": None, "images": None, "Plastic": None, "th": None}

In [22]:
def separateDatasetByObjectType(annotations):
  counts = getDictionaryWithObjectTypes()
  objects= getDictionaryWithObjectTypes()
  for key in counts.keys():
    objects[key] = [a for a in annotations if a['filename'].startswith(key)]
    counts[key] = len(objects[key])
  return counts, objects


train_count, train_objects = separateDatasetByObjectType(annotations_train) 
val_count, val_objects = separateDatasetByObjectType(annotations_val)

print("Train: ",train_count)
print("Val: ", val_count)

Train:  {'data': 9, 'Glass': 100, 'images': 24, 'Plastic': 124, 'th': 80}
Val:  {'data': 4, 'Glass': 50, 'images': 6, 'Plastic': 50, 'th': 30}
Test:  {'data': 4, 'Glass': 49, 'images': 6, 'Plastic': 49, 'th': 29}


## Trim half of train and val based on the object count:

In [ ]:
def trimDatasetToHalf(objects, counts_per_object, factor=0.5):
  trimmed_objects = getDictionaryWithObjectTypes()
  for key in objects:
    np.random.shuffle(objects[key])
    trimmed_objects[key] = objects[key][:int(counts_per_object[key]*factor)]
    print(key, " -> ", len(trimmed_objects[key]), " -> ", trimmed_objects[key][:5])
  return trimmed_objects

trimmed_train_objects = trimDatasetToHalf(train_objects, train_count, 0.5)
trimmed_train_objects = list(np.concatenate(list(trimmed_train_objects.values())).flat)

trimmed_val_objects = trimDatasetToHalf(val_objects, val_count, 0.5)
trimmed_val_objects = list(np.concatenate(list(trimmed_val_objects.values())).flat)

data  ->  3  ->  [{'fileref': '', 'size': 5580, 'filename': 'data(2).jpg', 'base64_img_data': '', 'file_attributes': {}, 'regions': {'0': {'shape_attributes': {'name': 'polygon', 'all_points_x': [120, 17, 38, 143, 120], 'all_points_y': [24, 29, 192, 178, 24]}, 'region_attributes': {'name': '1'}}, '1': {'shape_attributes': {'name': 'polygon', 'all_points_x': [120, 17, 38, 143, 120], 'all_points_y': [24, 29, 192, 178, 24]}, 'region_attributes': {'name': '1'}}}}, {'fileref': '', 'size': 3463, 'filename': 'data(37).jpg', 'base64_img_data': '', 'file_attributes': {}, 'regions': {'0': {'shape_attributes': {'name': 'polygon', 'all_points_x': [28, 53, 60, 136, 149, 178, 28], 'all_points_y': [1, 199, 204, 207, 201, 1, 1]}, 'region_attributes': {'name': '1'}}, '1': {'shape_attributes': {'name': 'polygon', 'all_points_x': [28, 53, 60, 136, 149, 178, 28], 'all_points_y': [1, 199, 204, 207, 201, 1, 1]}, 'region_attributes': {'name': '1'}}}}, {'fileref': '', 'size': 10387, 'filename': 'data(9).jpg',

## Save trimmed annotations to `data/train/region_data_train.json` and `data/val/region_data_val.json` respectively.

In [ ]:
def saveTrimmedAnnotations(mode, annotations):
  assert mode in ["train", "val"]
  dataset_dir = os.path.join(DATASET_DIR, mode)
  # annotation_file_path = os.path.join(dataset_dir, "region_data_"+mode+".json")
  annotation_file_path = os.path.join(dataset_dir, "region_data_"+mode+"_single_imgs.json")
  json.dump(annotations, open(annotation_file_path, 'w'))

saveTrimmedAnnotations("train", trimmed_train_objects)
saveTrimmedAnnotations("val", trimmed_val_objects)

## Test if trimmed files saved correctly:

In [ ]:
mode = "val"
dataset_dir = os.path.join(DATASET_DIR, mode)
annotation_file_path = os.path.join(dataset_dir, "region_data_"+mode+".json")
annotations = json.load(open(annotation_file_path))
annotations = list(annotations)
print(annotations)

[{'fileref': '', 'size': 5182, 'filename': 'data(62).jpg', 'base64_img_data': '', 'file_attributes': {}, 'regions': {'0': {'shape_attributes': {'name': 'polygon', 'all_points_x': [10, 19, 21, 34, 53, 73, 72, 71, 76, 88, 90, 76, 51, 24, 13, 10], 'all_points_y': [91, 101, 169, 182, 186, 179, 152, 149, 136, 132, 86, 77, 71, 79, 86, 91]}, 'region_attributes': {'name': '1'}}, '1': {'shape_attributes': {'name': 'polygon', 'all_points_x': [96, 113, 118, 117, 115, 104, 91, 83, 72, 74, 74, 74, 77, 96], 'all_points_y': [134, 139, 143, 153, 201, 212, 211, 210, 204, 173, 151, 145, 138, 134]}, 'region_attributes': {'name': '1'}}, '2': {'shape_attributes': {'name': 'polygon', 'all_points_x': [91, 104, 107, 115, 127, 148, 170, 190, 204, 205, 206, 211, 209, 193, 160, 134, 108, 91], 'all_points_y': [41, 59, 167, 176, 180, 185, 185, 176, 166, 99, 48, 37, 28, 20, 18, 20, 27, 41]}, 'region_attributes': {'name': '1'}}, '3': {'shape_attributes': {'name': 'polygon', 'all_points_x': [10, 19, 21, 34, 53, 73, 7

# Create test instances from half of val dataset that is unused:

In [ ]:
unseen_val_objects = getDictionaryWithObjectTypes()
for key in val_objects:
  np.random.shuffle(val_objects[key])
  unseen_val_objects[key] = val_objects[key][int(val_count[key]/2)+1:]
unseen_val_objects = list(np.concatenate(list(unseen_val_objects.values())).flat)
mode = "test"
dataset_dir = os.path.join(DATASET_DIR, mode)
annotation_file_path = os.path.join(dataset_dir, "region_data_"+mode+".json")
json.dump(unseen_val_objects, open(annotation_file_path, 'w'))


# Get images with only 1 object within them from the og annotation file!

i.e. images where all regions have the same polygon indices!

In [ ]:
mode = "val"
annotation_file_path = os.path.join("/content/drive/MyDrive/Colab Notebooks/COMP592_Transaprent_Obj_Detection/data/"+mode+"/region_data_"+mode+"_og.json")
annotations = json.load(open(annotation_file_path))
annotations = list(annotations.values())

annots_single = []
for a in annotations:
  polygons = [r['shape_attributes'] for r in a['regions'].values()]
  if(len(polygons) > 1 and (polygons.count(polygons[0]) == len(polygons))):
    annots_single.append(a)
print(len(annots_single), " images with just 1 transparent object found")
annotation_file_path = os.path.join("/content/drive/MyDrive/Colab Notebooks/COMP592_Transaprent_Obj_Detection/data/"+mode+"/region_data_"+mode+"_single_imgs.json")
json.dump(annots_single, open(annotation_file_path, 'w'))


182  images with just 1 transparent object found


In [ ]:
mode = "val"
annotation_file_path = os.path.join("/content/drive/MyDrive/Colab Notebooks/COMP592_Transaprent_Obj_Detection/data/"+mode+"/region_data_"+mode+"_single_imgs.json")
annotations = json.load(open(annotation_file_path))
print(len(annotations))

182
